In [1]:
import pandas as pd
import re
import json

In [2]:
df = pd.read_excel('../data/Depeche Mode Songs.xlsx')
df.head()

,Song,Album,Compilation?,Year,Writers,Unnamed: 5,Version Type,Notes
0,New Life,Speak and Spell,NaN,1981.0,Vince Clarke (age 20 - 21),NaN,NaN,NaN
1,I Sometimes Wish I Was Dead,Speak and Spell,NaN,1981.0,Vince Clarke (age 20 - 21),NaN,NaN,NaN
2,Puppets,Speak and Spell,NaN,1981.0,Vince Clarke (age 20 - 21),NaN,NaN,NaN
3,Boys Say Go!,Speak and Spell,NaN,1981.0,Vince Clarke (age 20 - 21),NaN,NaN,NaN
4,Nodisco,Speak and Spell,NaN,1981.0,Vince Clarke (age 20 - 21),NaN,NaN,NaN


In [3]:
# Drop the Compilation? column and the unnamed empty column E
df = df.drop(columns=['Compilation?', 'Unnamed: 5'], errors='ignore')

# Rename columns to clean JSON keys
df = df.rename(columns={'Version Type': 'Version_Type'})

df.head()

,Song,Album,Year,Writers,Version_Type,Notes
0,New Life,Speak and Spell,1981.0,Vince Clarke (age 20 - 21),NaN,NaN
1,I Sometimes Wish I Was Dead,Speak and Spell,1981.0,Vince Clarke (age 20 - 21),NaN,NaN
2,Puppets,Speak and Spell,1981.0,Vince Clarke (age 20 - 21),NaN,NaN
3,Boys Say Go!,Speak and Spell,1981.0,Vince Clarke (age 20 - 21),NaN,NaN
4,Nodisco,Speak and Spell,1981.0,Vince Clarke (age 20 - 21),NaN,NaN


In [4]:
def parse_writers(value):
    if pd.isna(value):
        return []
    writers = []
    for entry in str(value).split(';'):
        entry = entry.strip()
        match = re.match(r'^(.+?)\s*\(age\s*(.+?)\)$', entry)
        if match:
            writers.append({
                'name': match.group(1).strip(),
                'age': match.group(2).strip()
            })
        else:
            # Writer listed with no age (e.g. Andrew Phillpott)
            writers.append({'name': entry, 'age': None})
    return writers

df['Writers'] = df['Writers'].apply(parse_writers)
df[['Song', 'Writers']].head(10)

,Song,Writers
0,New Life,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
1,I Sometimes Wish I Was Dead,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
2,Puppets,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
3,Boys Say Go!,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
4,Nodisco,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
5,What's Your Name?,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
6,Photographic,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"
7,Tora! Tora! Tora!,"[{'name': 'Martin Gore', 'age': '19 - 20'}]"
8,Big Muff,"[{'name': 'Martin Gore', 'age': '19 - 20'}]"
9,Any Second Now (Voices),"[{'name': 'Vince Clarke', 'age': '20 - 21'}]"


In [5]:
string_cols = ['Song', 'Album', 'Version_Type', 'Notes']
for col in string_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

df.head()

,Song,Album,Year,Writers,Version_Type,Notes
0,New Life,Speak and Spell,1981.0,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]",NaN,NaN
1,I Sometimes Wish I Was Dead,Speak and Spell,1981.0,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]",NaN,NaN
2,Puppets,Speak and Spell,1981.0,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]",NaN,NaN
3,Boys Say Go!,Speak and Spell,1981.0,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]",NaN,NaN
4,Nodisco,Speak and Spell,1981.0,"[{'name': 'Vince Clarke', 'age': '20 - 21'}]",NaN,NaN


In [6]:
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Year'] = df['Year'].apply(lambda x: int(x) if pd.notna(x) else None)

df['Year'].head(20)

0     1981.0
1     1981.0
2     1981.0
3     1981.0
4     1981.0
5     1981.0
6     1981.0
7     1981.0
8     1981.0
9     1981.0
10    1981.0
11    1981.0
12    1981.0
13    1981.0
14    1981.0
15    1982.0
16    1982.0
17    1982.0
18    1982.0
19    1982.0
Name: Year, dtype: float64

In [9]:
df.to_json(
    '../data/depeche-songs.json',
    orient='records',
    force_ascii=False,
    indent=2
)
print("JSON exported successfully!")

JSON exported successfully!


In [10]:
with open('../data/depeche-songs.json', 'r') as f:
    print(f.read(500))

[
  {
    "Song":"New Life",
    "Album":"Speak and Spell",
    "Year":1981.0,
    "Writers":[
      {
        "name":"Vince Clarke",
        "age":"20 - 21"
      }
    ],
    "Version_Type":null,
    "Notes":null
  },
  {
    "Song":"I Sometimes Wish I Was Dead",
    "Album":"Speak and Spell",
    "Year":1981.0,
    "Writers":[
      {
        "name":"Vince Clarke",
        "age":"20 - 21"
      }
    ],
    "Version_Type":null,
    "Notes":null
  },
  {
    "Song":"Puppets",
    "Album":"Spea
